# 04 — ML Return Prediction Models

Extends the factor-based approach to machine learning models for
cross-sectional return prediction.

## Models

| # | Model | HP Tuning | Key Idea |
|---|-------|-----------|----------|
| 1 | Linear Regression | — | Baseline OLS |
| 2 | RBF Kernel Ridge | **Optuna** (80 trials) | Nystroem approximation + Ridge; tuning alpha, gamma, n_components |
| 3 | Random Forest | **Optuna** (100 trials) | Ensemble of trees; tuning depth, n_trees, min_leaf, max_features |
| 4 | Deep Neural Network | **Optuna** (50 trials) | Keras NN with L1 + Dropout; tuning 6 HPs jointly |
| 5 | AdaBoost | **Optuna** (100 trials) | Boosted decision trees; tuning 3 HPs |
| 6 | Max Sharpe Regression | — | Custom PyTorch loss maximizing portfolio Sharpe |
| 7 | IPCA | — | Instrumented PCA — latent factor model |

## HP Tuning Approach

All tunable models (RBF Kernel Ridge, Random Forest, DNN, AdaBoost) use
**[Optuna](https://doi.org/10.1145/3292500.3330701)** — Bayesian
optimization with the TPE (Tree-structured Parzen Estimator) sampler
(Akiba et al., KDD 2019). This is strictly more sample-efficient than grid
search: TPE models the objective function surface and focuses trials on
promising HP regions automatically — the same concept as iterative
coarse-to-fine refinement, but principled and adaptive.

All HP tuning uses **5-fold `TimeSeriesSplit`** to prevent look-ahead bias.

## Evaluation

Each model is evaluated on:
- OOS MSE, R², sign accuracy
- Long/short decile spread portfolio (Sharpe, drawdown, hit rate)

In [ ]:
# ── Environment setup (run once) ──────────────────────────────────────
import sys, os, subprocess

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_PATH = "/content/drive/MyDrive/quant-trading-experiments"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_PATH}[dev,ml]"])

    JKP_CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/CPSC 381-581: Machine Learning/Final project/jkp_data.csv"
else:
    JKP_CSV_PATH = "/Users/abhin/Library/CloudStorage/GoogleDrive-abhinavkeshri1999@gmail.com/My Drive/Colab Notebooks/CPSC 381-581: Machine Learning/Final project/jkp_data.csv"

print(f"Data path: {JKP_CSV_PATH}")
print(f"Exists: {os.path.exists(JKP_CSV_PATH)}")

try:
    import quant_trading
    print(f"quant_trading v{quant_trading.__version__} imported OK")
except ImportError:
    print("quant_trading not importable yet — restart kernel and run this cell again")

In [ ]:
import json
import os
import pickle
import gc
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

from quant_trading.data import (
    load_jkp_csv, winsorize_returns, prepare_ml_features,
    TARGET_COL, NUMERIC_FEATURES_ML, CATEGORICAL_FEATURES,
)
from quant_trading.signals import generate_positions
from quant_trading.portfolio import calculate_portfolio_returns
from quant_trading.evaluation import (
    evaluate_model_performance, calculate_max_drawdown,
    oos_r2, sort_ret_eq_wgt,
)

## 1. Data Loading, Preprocessing & Save to Disk

Load JKP data, winsorize returns, prepare ML features, split into
train/test, and **save to disk as parquet files**. This lets us free the
raw 8.9M-row DataFrame from memory (~10+ GB) and load only what's needed
for each model.

After this section runs once, subsequent cells load from the saved files.
You can restart the kernel and skip straight to any model section.

In [ ]:
SPLIT_DIR = os.path.join(os.path.dirname(JKP_CSV_PATH), "ml_splits")
RESULTS_DIR = os.path.join(os.path.dirname(JKP_CSV_PATH), "ml_return_prediction")
PREDICTIONS_DIR = os.path.join(RESULTS_DIR, "predictions")
SUMMARY_DIR = os.path.join(RESULTS_DIR, "summary")
MODELS_DIR = os.path.join(RESULTS_DIR, "models")
TRAIN_END_DATE = "2015-12-31"
TEST_START_DATE = "2016-01-01"

for path in [RESULTS_DIR, PREDICTIONS_DIR, SUMMARY_DIR, MODELS_DIR]:
    os.makedirs(path, exist_ok=True)

if os.path.exists(os.path.join(SPLIT_DIR, "X_train.parquet")):
    print(f"Saved splits found in {SPLIT_DIR} — skipping data prep.")
    print("Delete that folder to regenerate from scratch.")
else:
    print("Preparing data and saving train/test splits to disk...")
    os.makedirs(SPLIT_DIR, exist_ok=True)

    # Load and preprocess
    df_jkp = load_jkp_csv(JKP_CSV_PATH)
    df_jkp = winsorize_returns(df_jkp, lower=0.05, upper=0.95)
    df_jkp = df_jkp.loc[:, ~df_jkp.columns.duplicated(keep="first")]

    X_all, meta, y_all = prepare_ml_features(
        df_jkp,
        numeric_features=NUMERIC_FEATURES_ML,
        categorical_features=CATEGORICAL_FEATURES,
        target_col=TARGET_COL,
    )

    # Split
    train_mask = meta["month_date"] <= pd.Timestamp(TRAIN_END_DATE)
    test_mask = meta["month_date"] >= pd.Timestamp(TEST_START_DATE)

    # Build test_df with market_equity for portfolio evaluation
    test_df = meta.loc[test_mask].copy()
    test_df = test_df.merge(
        df_jkp[["id", "month_date", "market_equity"]].drop_duplicates(),
        on=["id", "month_date"], how="left",
    )

    # Save to parquet (fast, compact, preserves dtypes)
    X_all.loc[train_mask].to_parquet(os.path.join(SPLIT_DIR, "X_train.parquet"))
    X_all.loc[test_mask].to_parquet(os.path.join(SPLIT_DIR, "X_test.parquet"))
    y_all.loc[train_mask].to_frame().to_parquet(os.path.join(SPLIT_DIR, "y_train.parquet"))
    y_all.loc[test_mask].to_frame().to_parquet(os.path.join(SPLIT_DIR, "y_test.parquet"))
    meta.loc[train_mask].to_parquet(os.path.join(SPLIT_DIR, "meta_train.parquet"))
    test_df.to_parquet(os.path.join(SPLIT_DIR, "test_df.parquet"))

    # Save feature names
    pd.Series(list(X_all.columns)).to_csv(os.path.join(SPLIT_DIR, "features.csv"), index=False)

    print(f"Saved: train={train_mask.sum():,}, test={test_mask.sum():,}, features={X_all.shape[1]}")

    # Free all large objects
    del df_jkp, X_all, meta, y_all, test_df, train_mask, test_mask
    gc.collect()
    print("Memory freed.")

In [ ]:
# ── Load train/test from disk ──────────────────────────────────────────
# Only training data is loaded now. Test data is loaded later when needed.
# This keeps peak memory usage low.

X_train = pd.read_parquet(os.path.join(SPLIT_DIR, "X_train.parquet"))
y_train = pd.read_parquet(os.path.join(SPLIT_DIR, "y_train.parquet")).iloc[:, 0]
features = list(X_train.columns)

print(f"Train loaded: {len(X_train):,} rows × {len(features)} features")
print(f"Memory: ~{X_train.memory_usage(deep=True).sum() / 1e9:.1f} GB")

## Evaluation helpers

`evaluate_and_cleanup` loads test data from disk, runs prediction +
evaluation + portfolio analysis, then **frees test data from memory**.
Each model section calls this at the end so only training data stays resident.

In [ ]:
def evaluate_portfolio(df_eval, pred_col, model_name, summary_meta=None):
    """Evaluate long/short portfolio performance across percentile/weight grids."""
    pct_pairs = [(0.3, 0.7), (0.1, 0.9), (0.05, 0.95), (0.01, 0.99)]
    weight_schemes = ["equal", "char_rank_weighted", "char_minmax_weighted"]
    results = []
    summary_meta = summary_meta or {}

    for lower, upper in pct_pairs:
        positions = generate_positions(
            df_eval, pred_col, rebal_period=1,
            lower_pct=lower, upper_pct=upper, long_high=True,
        )
        for scheme in weight_schemes:
            try:
                port_rets = calculate_portfolio_returns(
                    positions, df_eval, ret_col=TARGET_COL,
                    weight_scheme=scheme, weight_col="market_equity",
                    char_col=pred_col, max_weight_per_leg=None,
                )
                spreads = port_rets["Spread"].dropna()
                if len(spreads) == 0:
                    continue
                ann_ret = spreads.mean() * 12
                ann_vol = spreads.std() * np.sqrt(12)
                row = {
                    "Model Name": model_name,
                    "Prediction Column": pred_col,
                    "Lower Pct": lower,
                    "Upper Pct": upper,
                    "Weight Scheme": scheme,
                    "Ann Ret (%)": ann_ret * 100,
                    "Ann Vol (%)": ann_vol * 100,
                    "Max DD (%)": calculate_max_drawdown(spreads) * 100,
                    "Sharpe": ann_ret / ann_vol if ann_vol > 0 else 0,
                    "Hit Rate (%)": (spreads > 0).mean() * 100,
                }
                row.update(summary_meta)
                results.append(row)
            except Exception as e:
                print(f"Error: {lower}-{upper} {scheme}: {e}")

    res_df = pd.DataFrame(results)
    print(f"\nPortfolio Evaluation for {pred_col}:")
    display(res_df.round(2))
    return res_df


def save_predictions_table(test_df, model_key, model_name, pred_col):
    """Save stock-month predictions for the held-out test sample only."""
    preds_out = test_df[["id", "month_date", TARGET_COL, pred_col, "market_equity"]].copy()
    preds_out = preds_out.rename(columns={
        TARGET_COL: "actual_return",
        pred_col: "predicted_return",
    })
    preds_out["model_name"] = model_name
    preds_out["prediction_column"] = pred_col
    preds_out["sample"] = "test"
    preds_path = os.path.join(PREDICTIONS_DIR, f"predictions_{model_key}.parquet")
    preds_out.to_parquet(preds_path, index=False)
    print(f"Saved test-only predictions to {preds_path}")
    return preds_path


def save_summary_table(summary_df, model_key):
    """Save portfolio summary for one model."""
    summary_path = os.path.join(SUMMARY_DIR, f"summary_{model_key}.parquet")
    summary_df.to_parquet(summary_path, index=False)
    print(f"Saved portfolio summary to {summary_path}")
    return summary_path


def evaluate_and_cleanup(model, model_name, pred_col, model_key, scaler=None, summary_meta=None):
    """Load test data, predict, evaluate, free test data from memory.

    Parameters
    ----------
    model : fitted model with .predict()
    model_name : str for display
    pred_col : column name to store predictions in test_df
    scaler : optional StandardScaler (for DNN/MSRR that need scaled input)
    summary_meta : optional metadata repeated into the saved summary rows
    """
    summary_meta = summary_meta or {}
    # Load test data from disk
    X_test = pd.read_parquet(os.path.join(SPLIT_DIR, "X_test.parquet"))
    test_df = pd.read_parquet(os.path.join(SPLIT_DIR, "test_df.parquet"))
    if not (test_df["month_date"] >= pd.Timestamp(TEST_START_DATE)).all():
        raise ValueError("test_df contains rows before TEST_START_DATE")
    print(f"Evaluating {model_name} on test sample only: {len(test_df):,} rows from {TEST_START_DATE} onward")

    # Predict
    if scaler is not None:
        X_input = scaler.transform(X_test).astype(np.float32)
    else:
        X_input = X_test.values

    preds = model.predict(X_input)
    if hasattr(preds, "flatten"):
        preds = preds.flatten()
    test_df[pred_col] = preds

    # Evaluate and persist outputs
    metrics = evaluate_model_performance(test_df[TARGET_COL], test_df[pred_col], model_name)
    summary_meta_full = {
        "Test Start Date": TEST_START_DATE,
        "Train End Date": TRAIN_END_DATE,
        "Sample": "test",
        "MSE": metrics["mse"],
        "R2": metrics["r2"],
        "Sign Accuracy": metrics["sign_acc"],
    }
    summary_meta_full.update(summary_meta)
    preds_path = save_predictions_table(test_df, model_key, model_name, pred_col)
    summary_df = evaluate_portfolio(test_df, pred_col, model_name, summary_meta=summary_meta_full)
    summary_path = save_summary_table(summary_df, model_key)

    # Free test data
    del X_test, test_df, X_input, preds
    gc.collect()
    print(f"  Test data freed from memory.")
    return {"metrics": metrics, "predictions_path": preds_path, "summary_path": summary_path}

## 2. Model 1 — Linear Regression (Baseline)

Ordinary least squares — no regularization, no tuning. Serves as the
minimum bar that more complex models should beat.

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
joblib.dump(lr_model, os.path.join(MODELS_DIR, "lr.joblib"))
evaluate_and_cleanup(
    lr_model,
    "Linear Regression",
    "pred_lr",
    model_key="lr",
    summary_meta={"Best Params": json.dumps({}), "Best CV Score": np.nan},
)
del lr_model; gc.collect()

---

# Background: RBF Kernel, Nystrom Approximation, Ridge, and Optuna

The next model (RBF Kernel Ridge) combines several ideas. This section
explains each piece and how they fit together.

## 1. RBF kernel intuition

Each data point is a location in feature space. The RBF (Radial Basis
Function) kernel measures how similar two points are:

$$k(x, x') = \exp(-\gamma \|x - x'\|^2)$$

- Very close points → value near 1.
- Far apart → value near 0.

A model using an RBF kernel behaves like a linear model in an
**infinite-dimensional** feature space (capturing all kinds of nonlinear
interactions like $x^2$, $xy$, etc.), but we never explicitly create those
terms.

**Mental model**: RBF = smooth similarity function that gives you very
flexible nonlinearity "for free."

## 2. Why Nystrom approximation

True kernel methods need an $n \times n$ kernel matrix (all pairwise
similarities), which costs $O(n^2)$ memory and $O(n^3)$ to solve — too
expensive when $n$ is large.

The **Nystrom method** approximates the kernel using a low-rank
representation:
1. Pick a small number of **landmark points** from your data.
2. Compute the RBF similarity of every point to these landmarks.
3. Use those similarities as explicit features.

**Mental model**: Nystrom = compress the implicit infinite feature space
into a manageable number of explicit features.

## 3. What `n_components` means

Suppose your original features are $(x_1, x_2, ..., x_p)$.

Setting `n_components = k` in `Nystroem`:
- Randomly selects $k$ **landmark samples** from the training data.
- For each data point, computes its RBF similarity to each landmark.
- Produces a **k-dimensional feature vector**:
  $$\phi(x) = (\phi_1(x), \dots, \phi_k(x))$$
  where each $\phi_j(x)$ is the similarity to landmark $j$.

So if `n_components = 200`, each sample becomes a 200-dimensional vector
of similarity scores — not polynomial powers like $x^{200}$.

**Rule of thumb**: larger `n_components` → better approximation to the
true RBF kernel (more expressive, more compute). Smaller → faster but
less expressive.

## 4. How landmarks are chosen

In scikit-learn's `Nystroem`, landmarks are selected by **uniform random
sampling** from the training data. They are real rows of your dataset
chosen at random — not learned or optimized.

For each sample, feature $j$ answers: *"How similar is this sample to
landmark $j$ under the RBF kernel?"*

**Mental model**: landmarks = randomly chosen reference points that define
the axes of your new feature space.

## 5. Ridge on top of Nystrom

```
Pipeline([
    ("nystroem", Nystroem(...)),  # maps X → Φ(X)
    ("ridge",    Ridge(...)),     # linear model on Φ(X)
])
```

After Nystrom, you have features $\phi_1, ..., \phi_k$. Ridge regression
fits:

$$y \approx \beta_0 + \beta_1 \phi_1 + \cdots + \beta_k \phi_k$$

This is **linear in $\phi$**, but **nonlinear in the original features**.

- **Nystrom**: builds a nonlinear feature map.
- **Ridge**: learns a linear combination of those nonlinear features with
  L2 regularization (prevents overfitting by shrinking coefficients).

## 6. What Optuna tunes here

Optuna searches three hyperparameters:

| HP | What it controls | Range | Scale |
|----|-----------------|-------|-------|
| `gamma` | RBF bandwidth. Small = broad/smooth, large = local/wiggly | [1e-4, 10] | log |
| `n_components` | Number of Nystrom landmarks (feature dimensionality) | [100, 500] | step=100 |
| `alpha` | Ridge regularization strength (coefficient shrinkage) | [1e-2, 1000] | log |

The `log=True` flag means Optuna samples **uniformly in log space** — so
it explores 0.001, 0.01, 0.1, 1, 10 in a balanced way, rather than
wasting budget on linear steps like 0.1, 0.2, 0.3.

## 7. Optuna vs grid search

**Grid search** picks a fixed grid (e.g., 5 values for gamma × 5 for
alpha = 25 combos) and evaluates every combination. It does not learn from
past results — it exhaustively sweeps the entire grid, including obviously
bad regions.

**Optuna** treats tuning as a sequential optimization problem. After each
trial, it uses the observed score to adjust where it samples next (via the
TPE sampler). First few trials spread out; later trials concentrate near
the best values seen so far. With the same budget of 25 evaluations,
Optuna typically finds a better configuration because it doesn't waste
trials on bad regions.

**Mental model**:
- Grid search: check every square on a chessboard, even if most are
  clearly losing.
- Optuna: explore broadly at first, then zoom in on the promising area.

## 8. Overall mental picture

1. **Original features**: $(x_1, x_2, ..., x_p)$
2. **Nystrom** (`n_components = k`): pick $k$ random landmarks, map each
   point to $k$ similarity scores → new feature space $\phi$
3. **Ridge**: fit a linear model on $\phi$ with L2 regularization
4. **Optuna**: efficiently search for the best (gamma, n_components, alpha)
   using adaptive Bayesian sampling instead of a rigid grid

Result: a **nonlinear but well-regularized model** whose complexity and
smoothness are controlled by hyperparameters that Optuna tunes efficiently.

---

## 3. Model 2 — RBF Kernel Ridge Regression

Nystroem approximation + Ridge regression — approximates a full kernel
Ridge at O(n × d²) instead of O(n³).

Hyperparameter tuning via **Optuna** (50 trials, 3-fold temporal CV).
HP search on 200k-row subsample; best model refit on full training data.

Search space:
- `alpha` (Ridge regularization): [1e-2, 1000] log-scale
- `gamma` (RBF kernel bandwidth): [1e-4, 10] log-scale
- `n_components` (Nystroem random features): [100, 500]

In [ ]:
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from quant_trading.tuning import tune_sklearn_model

# ── Subsample for HP search ───────────────────────────────────────────
# Nystrom + Ridge on 8M rows is feasible but slow for repeated CV.
# Subsample for tuning, refit best on full data.
TUNE_SUBSAMPLE = 200_000

if len(X_train) > TUNE_SUBSAMPLE:
    X_tune_rbf = X_train.values[-TUNE_SUBSAMPLE:]
    y_tune_rbf = y_train.values[-TUNE_SUBSAMPLE:]
    print(f"Subsampling {TUNE_SUBSAMPLE:,} / {len(X_train):,} rows for HP search")
else:
    X_tune_rbf = X_train.values
    y_tune_rbf = y_train.values


def rbf_ridge_objective(trial):
    return Pipeline([
        ("nystroem", Nystroem(
            kernel="rbf",
            gamma=trial.suggest_float("gamma", 1e-4, 10.0, log=True),
            n_components=trial.suggest_int("n_components", 100, 500, step=100),
            random_state=42,
        )),
        ("ridge", Ridge(
            alpha=trial.suggest_float("alpha", 1e-2, 1000.0, log=True),
        )),
    ])


print("RBF Kernel Ridge: Optuna tuning (3-fold temporal CV, 50 trials)...\n")
rbf_result = tune_sklearn_model(
    rbf_ridge_objective,
    X=X_tune_rbf, y=y_tune_rbf,
    n_trials=50,
    cv=3,
    scoring="neg_mean_squared_error",
)

# ── Refit best HP on full training data ────────────────────────────────
print(f"\nRefitting best RBF Kernel Ridge on full data ({len(X_train):,} rows)...")
best_rbf = Pipeline([
    ("nystroem", Nystroem(
        kernel="rbf",
        gamma=rbf_result["best_params"]["gamma"],
        n_components=rbf_result["best_params"]["n_components"],
        random_state=42,
    )),
    ("ridge", Ridge(alpha=rbf_result["best_params"]["alpha"])),
])
best_rbf.fit(X_train.values, y_train.values)
joblib.dump(best_rbf, os.path.join(MODELS_DIR, "rbf.joblib"))

evaluate_and_cleanup(
    best_rbf,
    "RBF Kernel Ridge",
    "pred_rbf",
    model_key="rbf",
    summary_meta={
        "Best Params": json.dumps(rbf_result["best_params"], sort_keys=True),
        "Best CV Score": rbf_result["best_score"],
    },
)
del best_rbf, rbf_result, X_tune_rbf, y_tune_rbf; gc.collect()

## 4. Model 3 — Random Forest

Ensemble of decision trees — captures nonlinear feature interactions
while averaging away individual-tree overfitting.

Hyperparameter tuning via **Optuna** (40 trials, 3-fold temporal CV).

**Compute note**: RF on ~8M rows is expensive. To keep HP tuning feasible,
we subsample the training data during Optuna search (200k rows), then
refit the best model on the full dataset. This is standard practice — HP
rankings are stable across subsample sizes.

Search space:
- `n_estimators`: [50, 300]
- `max_depth`: [3, 8]
- `min_samples_leaf`: [50, 500] (log-scale — large values prevent overfitting on panel data)
- `max_features`: [0.3, 1.0] (fraction of features per split)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from quant_trading.tuning import tune_sklearn_model

# ── HP search on a subsample (full data is too large for 40+ RF fits) ──
# Subsample 200k rows for tuning, preserving temporal order.
TUNE_SUBSAMPLE = 200_000

if len(X_train) > TUNE_SUBSAMPLE:
    # Take the LAST 200k rows (most recent data, preserves TimeSeriesSplit validity)
    X_tune = X_train.values[-TUNE_SUBSAMPLE:]
    y_tune = y_train.values[-TUNE_SUBSAMPLE:]
    print(f"Subsampling {TUNE_SUBSAMPLE:,} / {len(X_train):,} rows for HP search")
else:
    X_tune = X_train.values
    y_tune = y_train.values


def rf_objective(trial):
    return RandomForestRegressor(
        n_estimators=trial.suggest_int("n_estimators", 50, 300, step=50),
        max_depth=trial.suggest_int("max_depth", 3, 8),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 50, 500, log=True),
        max_features=trial.suggest_float("max_features", 0.3, 1.0),
        n_jobs=-1,
        random_state=42,
    )


print("Random Forest: Optuna tuning (3-fold temporal CV, 40 trials)...\n")
rf_result = tune_sklearn_model(
    rf_objective,
    X=X_tune, y=y_tune,
    n_trials=40,
    cv=3,
    scoring="neg_mean_squared_error",
)

# ── Refit best HP on full training data ────────────────────────────────
print(f"\nRefitting best RF on full training data ({len(X_train):,} rows)...")
best_rf = RandomForestRegressor(
    **{k: v for k, v in rf_result["best_params"].items()},
    n_jobs=-1,
    random_state=42,
)
best_rf.fit(X_train.values, y_train.values)
joblib.dump(best_rf, os.path.join(MODELS_DIR, "rf.joblib"))

evaluate_and_cleanup(
    best_rf,
    "Random Forest",
    "pred_rf",
    model_key="rf",
    summary_meta={
        "Best Params": json.dumps(rf_result["best_params"], sort_keys=True),
        "Best CV Score": rf_result["best_score"],
    },
)
del best_rf, rf_result, X_tune, y_tune; gc.collect()

## 5. Model 4 — Deep Neural Network (Keras)

Hyperparameter tuning via **Optuna** (30 trials, 3-fold temporal CV).
HP search runs on a 300k-row subsample; final model is trained on full data.

Search space (5 hyperparameters):
- L1 regularization: [1e-6, 1e-1] log-scale
- Learning rate: [1e-5, 1e-2] log-scale
- Dropout rate: [0.0, 0.5]
- Hidden layer 1 width: {32, 64, 96, 128}
- Hidden layer 2 width: {16, 32, 48, 64}

Key design choices vs. original:
- **Subsample for HP search**, refit on full data (8M rows × 50 trials is too slow)
- **Temporal validation split** for final training (not random `validation_split`)
- **Dropout** instead of L1 on biases (which harms BatchNorm)
- **Joint tuning** of L1 + learning rate + architecture (not L1 alone)
- **Early stopping patience=15** with `restore_best_weights=True` for final model

In [ ]:
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout, Input
from tensorflow.keras import regularizers
from quant_trading.tuning import tune_keras_nn

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)

N_FEATURES = X_train_scaled.shape[1]

# ── Subsample for HP search ───────────────────────────────────────────
# 50 trials × 3 folds × up to 50 epochs = 150 training runs.
# Subsample to keep each run fast; refit final model on full data.
TUNE_SUBSAMPLE = 300_000

if len(X_train_scaled) > TUNE_SUBSAMPLE:
    X_tune_nn = X_train_scaled[-TUNE_SUBSAMPLE:]
    y_tune_nn = y_train.values[-TUNE_SUBSAMPLE:]
    print(f"DNN: subsampling {TUNE_SUBSAMPLE:,} / {len(X_train_scaled):,} rows for HP search")
else:
    X_tune_nn = X_train_scaled
    y_tune_nn = y_train.values


def build_nn_optuna(trial):
    """Build a Keras NN with Optuna-suggested hyperparameters."""
    l1_reg = trial.suggest_float("l1_reg", 1e-6, 1e-1, log=True)
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    units_1 = trial.suggest_int("units_1", 32, 128, step=32)
    units_2 = trial.suggest_int("units_2", 16, 64, step=16)

    keras.backend.clear_session()
    model = keras.models.Sequential([
        Input(shape=(N_FEATURES,)),
        BatchNormalization(),
        Dense(units_1, activation="relu",
              kernel_regularizer=regularizers.L1(l1=l1_reg)),
        Dropout(dropout),
        BatchNormalization(),
        Dense(units_2, activation="relu",
              kernel_regularizer=regularizers.L1(l1=l1_reg)),
        Dropout(dropout),
        BatchNormalization(),
        Dense(1, activation="linear"),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="mse",
    )
    return model


# ── Optuna search on subsample ────────────────────────────────────────
print("DNN: Optuna tuning (3-fold temporal CV, 30 trials)...\n")
nn_result = tune_keras_nn(
    build_nn_optuna,
    X_train=X_tune_nn,
    y_train=y_tune_nn,
    n_trials=30,
    cv=3,
    epochs=50,
    batch_size=1024,
    patience=5,
)

# ── Final training on full data with temporal validation ──────────────
best = nn_result["best_params"]
n_train = int(len(X_train_scaled) * 0.8)

print(f"\nTraining final model with best HPs...")
print(f"  L1={best['l1_reg']:.5f}, LR={best['lr']:.5f}, "
      f"dropout={best['dropout']:.2f}, units=[{best['units_1']},{best['units_2']}]")
print(f"  Train: {n_train:,} rows | Temporal val: {len(X_train_scaled) - n_train:,} rows")

# Build final model with best params (manually, since we need a fresh trial)
keras.backend.clear_session()
final_nn = keras.models.Sequential([
    Input(shape=(N_FEATURES,)),
    BatchNormalization(),
    Dense(best["units_1"], activation="relu",
          kernel_regularizer=regularizers.L1(l1=best["l1_reg"])),
    Dropout(best["dropout"]),
    BatchNormalization(),
    Dense(best["units_2"], activation="relu",
          kernel_regularizer=regularizers.L1(l1=best["l1_reg"])),
    Dropout(best["dropout"]),
    BatchNormalization(),
    Dense(1, activation="linear"),
])
final_nn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=best["lr"]),
    loss="mse",
)

with tf.device("/CPU:0"):
    cb = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=15, restore_best_weights=True,
    )
    history = final_nn.fit(
        X_train_scaled[:n_train],
        y_train.iloc[:n_train].values.astype(np.float32),
        epochs=200,
        batch_size=best.get("batch_size", 512),
        validation_data=(
            X_train_scaled[n_train:],
            y_train.iloc[n_train:].values.astype(np.float32),
        ),
        callbacks=[cb],
        verbose=1,
    )
    best_epoch = np.argmin(history.history["val_loss"]) + 1
    print(f"  Best epoch: {best_epoch} "
          f"(val_loss={min(history.history['val_loss']):.6f})")

final_nn.save(os.path.join(MODELS_DIR, "dnn.keras"))
joblib.dump(scaler, os.path.join(MODELS_DIR, "dnn_scaler.joblib"))
evaluate_and_cleanup(
    final_nn,
    "Deep Neural Network",
    "pred_dnn",
    model_key="dnn",
    scaler=scaler,
    summary_meta={
        "Best Params": json.dumps(nn_result["best_params"], sort_keys=True),
        "Best CV Score": nn_result["best_score"],
    },
)

# Free DNN-specific objects
del final_nn, nn_result, X_train_scaled, X_tune_nn, y_tune_nn
keras.backend.clear_session()
gc.collect()

## 6. Model 5 — AdaBoost

Hyperparameter tuning via **Optuna** (40 trials, 3-fold temporal CV).
HP search on 200k-row subsample; best model refit on full training data.

AdaBoost is inherently sequential (each tree depends on the previous
one's errors), so it cannot be parallelized via `n_jobs`. This makes it
the slowest model per trial — hence the smaller trial budget and subsample.

Search space:
- `max_depth`: [1, 4] integer
- `n_estimators`: [50, 500] log-scale integer
- `learning_rate`: [1e-3, 1.0] log-scale float

In [ ]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from quant_trading.tuning import tune_sklearn_model

# ── Subsample for HP search ───────────────────────────────────────────
# AdaBoost is sequential (no n_jobs parallelism) and slow on large data.
# n_estimators=500 on 8M rows ≈ 10+ min per fit.
TUNE_SUBSAMPLE = 200_000

if len(X_train) > TUNE_SUBSAMPLE:
    X_tune_ada = X_train.values[-TUNE_SUBSAMPLE:]
    y_tune_ada = y_train.values[-TUNE_SUBSAMPLE:]
    print(f"Subsampling {TUNE_SUBSAMPLE:,} / {len(X_train):,} rows for HP search")
else:
    X_tune_ada = X_train.values
    y_tune_ada = y_train.values


def adaboost_objective(trial):
    return AdaBoostRegressor(
        estimator=DecisionTreeRegressor(
            max_depth=trial.suggest_int("max_depth", 1, 4),
            random_state=42,
        ),
        n_estimators=trial.suggest_int("n_estimators", 50, 500, log=True),
        learning_rate=trial.suggest_float("learning_rate", 1e-3, 1.0, log=True),
        random_state=42,
    )


print("AdaBoost: Optuna tuning (3-fold temporal CV, 40 trials)...\n")
ada_result = tune_sklearn_model(
    adaboost_objective,
    X=X_tune_ada, y=y_tune_ada,
    n_trials=40,
    cv=3,
    scoring="neg_mean_squared_error",
    n_jobs_cv=1,
)

# ── Refit best HP on full training data ────────────────────────────────
print(f"\nRefitting best AdaBoost on full data ({len(X_train):,} rows)...")
best_ada = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(
        max_depth=ada_result["best_params"]["max_depth"],
        random_state=42,
    ),
    n_estimators=ada_result["best_params"]["n_estimators"],
    learning_rate=ada_result["best_params"]["learning_rate"],
    random_state=42,
)
best_ada.fit(X_train.values, y_train.values)
joblib.dump(best_ada, os.path.join(MODELS_DIR, "adaboost.joblib"))

evaluate_and_cleanup(
    best_ada,
    "AdaBoost",
    "pred_adaboost",
    model_key="adaboost",
    summary_meta={
        "Best Params": json.dumps(ada_result["best_params"], sort_keys=True),
        "Best CV Score": ada_result["best_score"],
    },
)
del best_ada, ada_result, X_tune_ada, y_tune_ada; gc.collect()

## 7. Model 6 — Maximum Sharpe Ratio Regression (MSRR)

Custom PyTorch loss that directly maximizes the portfolio's Sharpe ratio
instead of minimizing MSE.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

# Scale training data for MSRR (DNN's scaler was freed)
msrr_scaler = StandardScaler()
X_train_sc = msrr_scaler.fit_transform(X_train).astype(np.float32)

X_train_tensor = torch.tensor(X_train_sc, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

# Month indices for grouping (needed for Sharpe loss)
meta_train = pd.read_parquet(os.path.join(SPLIT_DIR, "meta_train.parquet"))
train_months = meta_train["month_date"].astype("category").cat.codes.values
months_tensor = torch.tensor(train_months, dtype=torch.long)
del meta_train, X_train_sc; gc.collect()


class MaxSharpeRegression(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1, bias=True)

    def forward(self, x):
        return self.linear(x)


def sharpe_loss(predictions, targets, months):
    strategy_returns = predictions * targets
    unique_months = torch.unique(months)
    monthly_rets = torch.stack([
        strategy_returns[months == m].mean() for m in unique_months
    ])
    mean_ret = torch.mean(monthly_rets)
    std_ret = torch.std(monthly_rets) + 1e-6
    sharpe = (mean_ret / std_ret) * torch.sqrt(torch.tensor(12.0))
    return -sharpe  # minimize negative Sharpe


msrr_model = MaxSharpeRegression(X_train_tensor.shape[1])
optimizer = optim.Adam(msrr_model.parameters(), lr=0.01)

print("Training MSRR...")
for epoch in range(50):
    msrr_model.train()
    optimizer.zero_grad()
    preds = msrr_model(X_train_tensor)
    loss = sharpe_loss(preds, y_train_tensor, months_tensor)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1}/50 | Neg Sharpe: {loss.item():.4f}")

msrr_model.eval()
torch.save(msrr_model.state_dict(), os.path.join(MODELS_DIR, "msrr.pt"))
joblib.dump(msrr_scaler, os.path.join(MODELS_DIR, "msrr_scaler.joblib"))

# Load test, predict, evaluate, free
X_test = pd.read_parquet(os.path.join(SPLIT_DIR, "X_test.parquet"))
test_df = pd.read_parquet(os.path.join(SPLIT_DIR, "test_df.parquet"))
X_test_tensor = torch.tensor(msrr_scaler.transform(X_test).astype(np.float32))
with torch.no_grad():
    test_df["pred_msrr"] = msrr_model(X_test_tensor).numpy().flatten()
if not (test_df["month_date"] >= pd.Timestamp(TEST_START_DATE)).all():
    raise ValueError("test_df contains rows before TEST_START_DATE")
print(f"Evaluating MSRR on test sample only: {len(test_df):,} rows from {TEST_START_DATE} onward")
metrics_msrr = evaluate_model_performance(test_df[TARGET_COL], test_df["pred_msrr"], "MSRR")
save_predictions_table(test_df, "msrr", "MSRR", "pred_msrr")
summary_msrr = evaluate_portfolio(
    test_df,
    "pred_msrr",
    "MSRR",
    summary_meta={
        "Test Start Date": TEST_START_DATE,
        "Train End Date": TRAIN_END_DATE,
        "Sample": "test",
        "MSE": metrics_msrr["mse"],
        "R2": metrics_msrr["r2"],
        "Sign Accuracy": metrics_msrr["sign_acc"],
        "Best Params": json.dumps({"optimizer_lr": 0.01, "epochs": 50}, sort_keys=True),
        "Best CV Score": np.nan,
    },
)
save_summary_table(summary_msrr, "msrr")
del X_test, test_df, X_test_tensor, msrr_model, X_train_tensor, y_train_tensor, months_tensor
gc.collect()
print("  Test data freed from memory.")

## 8. Model 7 — IPCA (Instrumented PCA)

Instrumented PCA (Kelly, Pruitt, Su 2019) estimates latent factors whose
loadings are linear functions of observable firm characteristics.
Uses 3 latent factors with intercept. OOS prediction via `mean_factor=True`
(assumes test-period factors equal their historical average — conservative).

In [ ]:
# pip install ipca
from ipca import InstrumentedPCA

# IPCA requires MultiIndex (entity, time) DataFrames.
# Load train/test metadata and features from disk.
meta_train = pd.read_parquet(os.path.join(SPLIT_DIR, "meta_train.parquet"))
X_test = pd.read_parquet(os.path.join(SPLIT_DIR, "X_test.parquet"))
y_test = pd.read_parquet(os.path.join(SPLIT_DIR, "y_test.parquet")).iloc[:, 0]
test_df_meta = pd.read_parquet(os.path.join(SPLIT_DIR, "test_df.parquet"))

train_model = pd.concat([meta_train, X_train], axis=1)
test_model = pd.concat([test_df_meta[["id", "month_date", TARGET_COL]], X_test], axis=1)

X_train_ipca = train_model[["id", "month_date"] + features].set_index(["id", "month_date"])
y_train_ipca = train_model.set_index(["id", "month_date"])[TARGET_COL]
X_test_ipca = test_model[["id", "month_date"] + features].set_index(["id", "month_date"])

ipca_model = InstrumentedPCA(n_factors=3, intercept=True)
print("Training IPCA...")
ipca_model.fit(X_train_ipca, y_train_ipca)
with open(os.path.join(MODELS_DIR, "ipca.pkl"), "wb") as f:
    pickle.dump(ipca_model, f)

pred_ipca = ipca_model.predict(X=X_test_ipca, mean_factor=True)

# Load test_df for portfolio evaluation
test_df = pd.read_parquet(os.path.join(SPLIT_DIR, "test_df.parquet"))
test_df["pred_ipca"] = pred_ipca.values if hasattr(pred_ipca, "values") else pred_ipca

if not (test_df["month_date"] >= pd.Timestamp(TEST_START_DATE)).all():
    raise ValueError("test_df contains rows before TEST_START_DATE")
print(f"Evaluating IPCA on test sample only: {len(test_df):,} rows from {TEST_START_DATE} onward")
metrics_ipca = evaluate_model_performance(test_df[TARGET_COL], test_df["pred_ipca"], "IPCA")
save_predictions_table(test_df, "ipca", "IPCA", "pred_ipca")
summary_ipca = evaluate_portfolio(
    test_df,
    "pred_ipca",
    "IPCA",
    summary_meta={
        "Test Start Date": TEST_START_DATE,
        "Train End Date": TRAIN_END_DATE,
        "Sample": "test",
        "MSE": metrics_ipca["mse"],
        "R2": metrics_ipca["r2"],
        "Sign Accuracy": metrics_ipca["sign_acc"],
        "Best Params": json.dumps({"n_factors": 3, "intercept": True}, sort_keys=True),
        "Best CV Score": np.nan,
    },
)
save_summary_table(summary_ipca, "ipca")
del test_df; gc.collect()

## 9. IPCA Factor Analysis

Inspect the Gamma matrix (how each characteristic loads onto the latent
factors) and compute the in-sample tangency portfolio of the estimated
factor returns.

In [ ]:
# Gamma matrix: weights of each characteristic in factor construction
num_components = ipca_model.Gamma.shape[1]
col_names = (
    ["Intercept"] + [f"Factor_{i+1}" for i in range(ipca_model.n_factors)]
    if num_components > ipca_model.n_factors
    else [f"Factor_{i+1}" for i in range(ipca_model.n_factors)]
)

gamma_df = pd.DataFrame(ipca_model.Gamma, index=features, columns=col_names)
print("IPCA Gamma Matrix:")
display(gamma_df.style.background_gradient(cmap="coolwarm"))

# Tangency portfolio of IPCA factors
if ipca_model.Factors is not None:
    factor_rets = pd.DataFrame(ipca_model.Factors.T, columns=col_names)
    mean_factors = factor_rets.mean()
    cov_factors = factor_rets.cov()
    w_tangency = np.linalg.pinv(cov_factors).dot(mean_factors)
    w_tangency = w_tangency / np.sum(np.abs(w_tangency))

    print("\nTangency Portfolio Weights:")
    for name, w in zip(col_names, w_tangency):
        print(f"  {name}: {w:.2%}")

    tangency_returns = factor_rets.dot(w_tangency)
    ann_ret = tangency_returns.mean() * 12
    ann_vol = tangency_returns.std() * np.sqrt(12)
    print(f"\nIn-Sample Tangency: Return={ann_ret:.2%}, Vol={ann_vol:.2%}, Sharpe={ann_ret/ann_vol:.2f}")